In [ ]:
# ============================================================
#  Local RAG Q&A System – HTML Frontend + Flask Backend
#  No APIs, no Streamlit – runs entirely in Colab
# ============================================================

# --------------------------
# 1. Install dependencies
# --------------------------
!pip install -q flask pyngrok sentence-transformers faiss-cpu pypdf python-docx langchain langchain-text-splitters transformers torch

# --------------------------
# 2. Imports
# --------------------------
import os
import json
import tempfile
import threading
import subprocess
import numpy as np
from flask import Flask, request, jsonify, render_template_string
from pyngrok import ngrok
from sentence_transformers import SentenceTransformer
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pypdf
from docx import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from google.colab import files
from google.colab import userdata

# --------------------------
# 3. Helper: load document text
# --------------------------
def load_document(uploaded_file) -> str:
    file_extension = uploaded_file.name.split('.')[-1].lower()
    text = ""
    with tempfile.NamedTemporaryFile(delete=False, suffix=f".{file_extension}") as tmp:
        tmp.write(uploaded_file.getbuffer())
        tmp_path = tmp.name

    if file_extension == "pdf":
        reader = pypdf.PdfReader(tmp_path)
        for page in reader.pages:
            text += page.extract_text() or ""
    elif file_extension == "docx":
        doc = Document(tmp_path)
        for para in doc.paragraphs:
            text += para.text + "\n"
    else:  # txt, md
        with open(tmp_path, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()

    os.unlink(tmp_path)
    return text

# --------------------------
# 4. Local Vector Store (FAISS)
# --------------------------
class LocalVectorStore:
    def __init__(self, embedding_model_name: str = "all-MiniLM-L6-v2"):
        self.embedder = SentenceTransformer(embedding_model_name)
        self.dimension = self.embedder.get_sentence_embedding_dimension()
        self.index = None
        self.chunks = []

    def add_documents(self, texts: list[str]):
        embeddings = self.embedder.encode(texts, show_progress_bar=True)
        if self.index is None:
            self.index = faiss.IndexFlatL2(self.dimension)
        self.index.add(np.array(embeddings, dtype=np.float32))
        self.chunks.extend(texts)

    def search(self, query: str, k: int = 3):
        q_embedding = self.embedder.encode([query])
        distances, indices = self.index.search(np.array(q_embedding, dtype=np.float32), k)
        results = []
        for idx, dist in zip(indices[0], distances[0]):
            if idx != -1:
                results.append({"chunk": self.chunks[idx], "score": float(dist)})
        return results

# --------------------------
# 5. Local LLM (Phi-3 Mini)
# --------------------------
def load_local_llm():
    model_name = "microsoft/Phi-3-mini-4k-instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    return pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.3,
        top_p=0.9
    )

def generate_answer(llm_pipeline, query: str, context_chunks: list[str]) -> str:
    context = "\n\n".join([f"[{i+1}] {chunk}" for i, chunk in enumerate(context_chunks)])
    prompt = f"""You are a helpful assistant. Answer the question using ONLY the provided context.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""
    output = llm_pipeline(prompt)[0]['generated_text']
    answer = output.split("Answer:")[-1].strip()
    return answer

# --------------------------
# 6. Upload documents & build RAG components
# --------------------------
print("📁 Please upload your documents (PDF, DOCX, TXT):")
uploaded = files.upload()

all_text = ""
for filename, content in uploaded.items():
    print(f"Loading {filename}...")
    fake_file = type('FakeFile', (), {'name': filename, 'getbuffer': lambda: content})()
    text = load_document(fake_file)
    all_text += text + "\n\n"

if not all_text.strip():
    raise ValueError("No text extracted. Check your files.")

print(f"✅ Total extracted characters: {len(all_text)}")

# Chunking
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(all_text)
print(f"📄 Created {len(chunks)} chunks.")

# Build vector store
print("🧠 Building vector store with sentence-transformers...")
vector_store = LocalVectorStore()
vector_store.add_documents(chunks)
print("✅ Vector store ready.")

# Load local LLM (first time will download ~2.3 GB)
print("🤖 Loading local LLM (Phi-3 Mini) – this may take 2-3 minutes...")
llm = load_local_llm()
print("✅ LLM loaded.")

# --------------------------
# 7. Flask app with HTML frontend
# --------------------------
app = Flask(__name__)

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Local RAG Q&A System</title>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
            margin: 0;
            padding: 20px;
            min-height: 100vh;
        }
        .container {
            max-width: 900px;
            margin: 0 auto;
            background: white;
            border-radius: 20px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.1);
            overflow: hidden;
        }
        .header {
            background: #2c3e50;
            color: white;
            padding: 20px;
            text-align: center;
        }
        .header h1 {
            margin: 0;
            font-size: 1.8rem;
        }
        .header p {
            margin: 5px 0 0;
            opacity: 0.8;
        }
        .chat-area {
            padding: 20px;
            height: 400px;
            overflow-y: auto;
            background: #f9f9f9;
        }
        .message {
            margin-bottom: 15px;
            display: flex;
            flex-direction: column;
        }
        .user-message {
            align-items: flex-end;
        }
        .assistant-message {
            align-items: flex-start;
        }
        .bubble {
            max-width: 80%;
            padding: 10px 15px;
            border-radius: 18px;
            line-height: 1.4;
        }
        .user-bubble {
            background: #007bff;
            color: white;
            border-bottom-right-radius: 4px;
        }
        .assistant-bubble {
            background: #e9ecef;
            color: #333;
            border-bottom-left-radius: 4px;
        }
        .sources {
            font-size: 0.8rem;
            color: #6c757d;
            margin-top: 5px;
            cursor: pointer;
        }
        .input-area {
            display: flex;
            padding: 20px;
            background: white;
            border-top: 1px solid #ddd;
        }
        #query {
            flex: 1;
            padding: 12px;
            border: 1px solid #ccc;
            border-radius: 25px;
            font-size: 1rem;
            outline: none;
        }
        #query:focus {
            border-color: #007bff;
        }
        button {
            margin-left: 10px;
            padding: 12px 24px;
            background: #007bff;
            color: white;
            border: none;
            border-radius: 25px;
            cursor: pointer;
            font-weight: bold;
        }
        button:hover {
            background: #0056b3;
        }
        .loader {
            display: inline-block;
            width: 20px;
            height: 20px;
            border: 3px solid #f3f3f3;
            border-top: 3px solid #007bff;
            border-radius: 50%;
            animation: spin 1s linear infinite;
        }
        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }
        .footer {
            text-align: center;
            padding: 15px;
            font-size: 0.8rem;
            background: #f1f1f1;
            color: #555;
        }
    </style>
</head>
<body>
<div class="container">
    <div class="header">
        <h1>📄 Local RAG Q&A System</h1>
        <p>Ask questions about your documents – all processing happens inside Colab</p>
    </div>
    <div class="chat-area" id="chatArea">
        <div class="message assistant-message">
            <div class="bubble assistant-bubble">
                👋 Hello! I have indexed your documents. Ask me anything about them.
            </div>
        </div>
    </div>
    <div class="input-area">
        <input type="text" id="query" placeholder="Type your question here..." autocomplete="off">
        <button id="sendBtn">Send</button>
    </div>
    <div class="footer">
        🔒 Fully local: sentence-transformers + FAISS + Microsoft Phi-3 Mini &nbsp;|&nbsp; No data leaves this notebook
    </div>
</div>

<script>
    const chatArea = document.getElementById('chatArea');
    const queryInput = document.getElementById('query');
    const sendBtn = document.getElementById('sendBtn');

    function addMessage(role, text, sources = null) {
        const messageDiv = document.createElement('div');
        messageDiv.className = `message ${role}-message`;
        const bubble = document.createElement('div');
        bubble.className = `bubble ${role === 'user' ? 'user-bubble' : 'assistant-bubble'}`;
        bubble.innerHTML = text.replace(/\\n/g, '<br>');
        messageDiv.appendChild(bubble);

        if (sources && sources.length > 0) {
            const sourcesDiv = document.createElement('div');
            sourcesDiv.className = 'sources';
            sourcesDiv.innerHTML = `📚 Sources: ${sources.map(s => `[score ${s.score.toFixed(2)}]`).join(', ')}`;
            sourcesDiv.onclick = () => {
                alert(sources.map((s, i) => `Source ${i+1}:\\n${s.chunk.substring(0, 300)}...`).join('\\n\\n'));
            };
            messageDiv.appendChild(sourcesDiv);
        }

        chatArea.appendChild(messageDiv);
        chatArea.scrollTop = chatArea.scrollHeight;
    }

    async function askQuestion() {
        const query = queryInput.value.trim();
        if (!query) return;

        // Disable UI
        sendBtn.disabled = true;
        queryInput.disabled = true;

        // Show user message
        addMessage('user', query);
        queryInput.value = '';

        // Show loading indicator
        const loadingDiv = document.createElement('div');
        loadingDiv.className = 'message assistant-message';
        loadingDiv.innerHTML = '<div class="bubble assistant-bubble"><div class="loader"></div> Thinking...</div>';
        chatArea.appendChild(loadingDiv);
        chatArea.scrollTop = chatArea.scrollHeight;

        try {
            const response = await fetch('/ask', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ query: query })
            });
            const data = await response.json();

            // Remove loading indicator
            chatArea.removeChild(loadingDiv);

            if (data.error) {
                addMessage('assistant', `❌ Error: ${data.error}`);
            } else {
                addMessage('assistant', data.answer, data.sources);
            }
        } catch (err) {
            chatArea.removeChild(loadingDiv);
            addMessage('assistant', `❌ Network error: ${err.message}`);
        } finally {
            sendBtn.disabled = false;
            queryInput.disabled = false;
            queryInput.focus();
        }
    }

    sendBtn.addEventListener('click', askQuestion);
    queryInput.addEventListener('keypress', (e) => {
        if (e.key === 'Enter') askQuestion();
    });
</script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/ask', methods=['POST'])
def ask():
    data = request.get_json()
    query = data.get('query', '').strip()
    if not query:
        return jsonify({'error': 'Empty query'}), 400

    # Retrieve
    retrieved = vector_store.search(query, k=3)
    if not retrieved:
        return jsonify({'answer': 'No relevant information found in your documents.', 'sources': []})

    context_chunks = [r['chunk'] for r in retrieved]
    # Generate
    answer = generate_answer(llm, query, context_chunks)

    # Return with source info (chunks and scores)
    sources = [{'chunk': r['chunk'][:200] + '...', 'score': r['score']} for r in retrieved]
    return jsonify({'answer': answer, 'sources': sources})

# --------------------------
# 8. Start Flask with ngrok
# --------------------------
# Kill previous processes
!pkill -f flask 2>/dev/null
!pkill -f ngrok 2>/dev/null

# Set ngrok auth token (optional, but recommended to avoid session limits)
NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Start Flask in a background thread
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask)
thread.start()

# Open ngrok tunnel
public_url = ngrok.connect(5000)
print(f"\n🎉 HTML RAG system is live!\n👉 Click here: {public_url}\n")
print("Note: The first query will be slower because the LLM needs to warm up.")

ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation on testing for
information about how to avoid this problem.


📁 Please upload your documents (PDF, DOCX, TXT):
